In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [12]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm = ChatOpenAI()

def chat_node(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages': [response]}


In [15]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chat = graph.compile(checkpointer=checkpointer)

initial_state = {'messages': [HumanMessage(content="Hello, what is capital of india ?")]}

thread_id = '1'
config = {'configurable': {'thread_id': thread_id}}
chat.invoke(initial_state,config=config)

{'messages': [HumanMessage(content='Hello, what is capital of india ?', additional_kwargs={}, response_metadata={}, id='eac74e02-d842-462b-84fd-024ea6c162fb'),
  AIMessage(content='The capital of India is New Delhi.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 15, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DT4TUZtV6gBgvEXFd7zezBWTKeVRk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d772f-524f-76f2-8def-3df2231540f2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output

In [16]:
while True:

    user_message = input("User: ")
    if user_message.lower() in ['exit','quit']:
        print("Exiting the chat.")
        break

    response = chat.invoke({'messages': [HumanMessage(content=user_message)]},config=config)
    print(f"Assistant: {response['messages'][-1].content}")

Assistant: Hello Nikhil! How can I assist you today?
Assistant: Your name is Nikhil.
Exiting the chat.
